# Dataset manifests

Prepare local NumPy datasets with scene isolation and explicit contracts.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Manifest contract

Images are HWC uint8 RGB; labels are aligned integer HW arrays. Paths are confined to the manifest directory.

In [ ]:
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.data import RoadScenes
from autonomous_vehicles.perception.dataset import ManifestDataset, load_manifest, manifest_summary

## Build synthetic fixture files

Each sample has a unique ID and scene. No external data or licensing assumptions are introduced.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    samples = []
    for split in ("train", "validation", "test"):
        dataset = RoadScenes(3, 32, seed=42, split=split)
        for index in range(len(dataset)):
            image, mask = dataset[index]
            identifier = f"{split}-{index}"
            np.save(
                root / f"{identifier}-image.npy",
                (image.permute(1, 2, 0).numpy() * 255).astype(np.uint8),
            )
            np.save(root / f"{identifier}-mask.npy", mask.numpy())
            samples.append(
                {
                    "identifier": identifier,
                    "scene": identifier,
                    "split": split,
                    "image": f"{identifier}-image.npy",
                    "mask": f"{identifier}-mask.npy",
                }
            )
    manifest = root / "manifest.json"
    manifest.write_text(json.dumps({"format_version": 1, "samples": samples}))
    records = load_manifest(manifest)
    summary = manifest_summary(records)
    loaded_image, loaded_mask = ManifestDataset(manifest, "test")[0]
print(summary)

## Inspect loaded tensors

The loader normalizes uint8 images and preserves integer labels.

In [ ]:
assert loaded_image.shape == (3, 32, 32)
assert loaded_mask.shape == (32, 32)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(loaded_image.permute(1, 2, 0))
axes[1].imshow(loaded_mask, vmin=0, vmax=2)
axes[0].set_title("Loaded RGB")
axes[1].set_title("Loaded labels")
plt.show()

## Scene split policy

Frames from the same scene must remain in one split, even if file names differ.

In [ ]:
scene_splits = {}
for record in records:
    scene_splits.setdefault(record.scene, set()).add(record.split)
assert all(len(splits) == 1 for splits in scene_splits.values())
print("Scene groups:", len(scene_splits))
print("Train samples:", summary["splits"]["train"])
print("Validation samples:", summary["splits"]["validation"])
print("Test samples:", summary["splits"]["test"])

## Adapting real datasets

Before real data ingestion, map labels, preserve scene IDs, document consent/licensing and define units.

In [ ]:
contract = {
    "image": "HWC uint8, RGB, no hidden channel reversal",
    "mask": "HW integer class IDs 0 through classes-1",
    "scene": "stable sequence identity for leakage prevention",
    "split": "train, validation, or test",
    "paths": "relative to manifest; no escape via parent or symlink",
}
for field, requirement in contract.items():
    print(f"{field}: {requirement}")

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.